In [ ]:
import sklearnex
sklearnex.patch_sklearn()
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler

In [ ]:
# Filter outliers using IQR
def filter_outliers(df, column):
    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    df[column] = df[column].where((df[column] >= lower_bound) & (df[column] <= upper_bound), None)
    return df

In [ ]:
from sklearn.model_selection import train_test_split
df = pd.read_csv(Path("e1_nutrients.csv"))
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

In [ ]:
train_filtered = train_df.copy()
for depth in train_filtered["Depth"].unique():
    mask = train_filtered["Depth"] == depth
    depth_subset = train_filtered.loc[mask].copy()
    for column in train_filtered.columns:
        depth_subset = filter_outliers(depth_subset, column)
    train_filtered.loc[mask] = depth_subset
train_clean = train_filtered.dropna()

scaler = MinMaxScaler()
y_scaler = MinMaxScaler()
x_train = scaler.fit_transform(train_clean.drop(columns=["Depth"]))
y_train = y_scaler.fit_transform(train_clean[["Depth"]]).ravel()
x_test = scaler.transform(test_df.drop(columns=["Depth"]))
y_test = y_scaler.transform(test_df[["Depth"]]).ravel()

df_filtered_minmax = train_filtered
df_scaled_minmax = train_clean.copy()
df_scaled_minmax.loc[:, df_scaled_minmax.columns != "Depth"] = x_train

In [ ]:
# Show filtering and scaling results to ensure they are working
plot_columns = list(df.columns[1:])
fig, axes = plt.subplots(ncols=2, nrows=3, figsize=(10, 12), sharex=True)
axes = axes.flatten()

for ax, column in zip(axes, plot_columns):
    ax.plot(df_filtered_minmax['Depth'], df_filtered_minmax[column], 'o', markersize=5, label='Filtered')
    ax.plot(df_scaled_minmax['Depth'] + 1, df_scaled_minmax[column], 'x', markersize=5, alpha=0.5, label='Scaled')
    ax.set_xlabel("Depth")
    ax.set_ylabel(column.capitalize())
    ax.set_title(f"{column.capitalize()} vs Depth")

axes[-1].axis("off")

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=2)
plt.tight_layout(rect=[0, 0, 1, 1])
plt.show()

In [ ]:
# try to find best paramaters using grid search
from sklearn.model_selection import GridSearchCV
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score, mean_squared_error
mlp = MLPRegressor(max_iter=5000, random_state=42, n_iter_no_change=20)
param_grid = [
    {
        'solver': ['adam'],
        'hidden_layer_sizes': [(100,)],
        'learning_rate': ['constant'],
        'activation': ['relu'],
        'alpha': [0.0001],
        'learning_rate_init': [0.001],
        'validation_fraction': [0.1],
        'early_stopping': [False]
    },
    {
        'solver': ['adam'],
        'hidden_layer_sizes': [(64,), (128,), (64,64), (128,64)],
        'activation': ['relu'],
        'alpha': [0.0001, 0.001, 0.01],
        'learning_rate_init': [1e-4, 1e-3, 1e-2],
        'validation_fraction': [0.15, 0.2],
        'early_stopping': [True]
    },
]
grid_search = GridSearchCV(estimator=mlp, param_grid=param_grid, cv=5, n_jobs=8, verbose=2, scoring='neg_mean_squared_error', return_train_score=True)
grid_search.fit(x_train, y_train)

print("Best parameters found: ", grid_search.best_params_)
print("MSE test: ", mean_squared_error(y_test, grid_search.best_estimator_.predict(x_test)))

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestRegressor

regressors = []
regressors.append(("RandomForestRegressor-default", RandomForestRegressor(random_state=42)))
regressors.append(("LinearRegression-default", LinearRegression()))
regressors.append(("MLPRegressor-default", MLPRegressor(random_state=42)))
# params from grid search with best cv mse
regressors.append(("MLPRegressor-tunedadam1", MLPRegressor(max_iter=2000, hidden_layer_sizes=(128,64), validation_fraction=0.2, learning_rate_init=0.001, alpha=0.001, early_stopping=True, random_state=42)))

scores = []

for regressor_name, regressor in regressors:
    regressor.fit(x_train, y_train)
    train_preds = regressor.predict(x_train)
    test_preds = regressor.predict(x_test)
    train_r2 = r2_score(y_train, train_preds)
    test_r2 = r2_score(y_test, test_preds)
    train_mse = mean_squared_error(y_train, train_preds)
    test_mse = mean_squared_error(y_test, test_preds)
    scores.append((regressor_name, train_r2, test_r2, train_mse, test_mse))

pd.DataFrame(scores, columns=["model", "train_r2", "test_r2", "train_mse", "test_mse"]).sort_values("test_mse") 

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

regressor_names = [score[0] for score in scores]
r2_values = [score[1] for score in scores]
ms_values = [score[2] for score in scores]

ax1.bar(range(len(regressor_names)), r2_values)
ax1.set_xticks(range(len(regressor_names)))
ax1.set_xticklabels(regressor_names, rotation=45, ha='right')
ax1.set_ylabel('R² Score')
ax1.set_title('R² Score by Regressor')

ax2.bar(range(len(regressor_names)), ms_values)
ax2.set_xticks(range(len(regressor_names)))
ax2.set_xticklabels(regressor_names, rotation=45, ha='right')
ax2.set_ylabel('Mean Squared Error')
ax2.set_title('Mean Squared Error by Regressor')

plt.tight_layout()
plt.show()